# Band Director Agent — Demo

This notebook demonstrates the Band Director Agent: a natural language interface that lets a band director manage their instrument inventory, student roster, and music library without writing any SQL.

**Pipeline:** User prompt $\to$ Planner (LLM) $\to$ Generator $\to$ Validator $\to$ Executor $\to$ Formatter (LLM) $\to$ Natural language response

---

## Setup

Install dependencies, configure credentials, and import the pipeline.

> **Colab:** Add `ANTHROPIC_API_KEY` and `DATABASE_URL` to your Colab Secrets before running.

In [ ]:
%pip install -q anthropic psycopg2-binary python-dotenv

In [3]:
import os
import sys

# Colab: load credentials from Secrets
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    os.environ["DATABASE_URL"]      = userdata.get("DATABASE_URL")
    print("Credentials loaded from Colab Secrets.")
except Exception:
    # Local: credentials loaded from .env by the pipeline modules
    print("Running locally — credentials will be loaded from .env")

# Add src/ to path (adjust if your repo is cloned elsewhere)
REPO_SRC = os.path.normpath(os.path.join(os.getcwd(), "..", "src"))
sys.path.insert(0, REPO_SRC)
print(f"Source path: {REPO_SRC}")

Running locally — credentials will be loaded from .env
Source path: /Users/Chris/Developer/GitHub/school/CSCI-264/Band-Class-Agent/src


In [ ]:
from main import chat
from database.connection import get_connection

print("Pipeline imported successfully.")

---

## Single INSERT

Ask the agent to add one student to the roster. The Planner classifies the intent and extracts the student's name and grade. The Generator builds a parameterized `INSERT`. The Validator checks the grade range. The Executor writes the row. The Formatter replies in plain English.

In [ ]:
response = chat("Add Emma Rodriguez, a 10th grader, to the student list.")
print(response)

### Verify with a SELECT

Query the roster to confirm the row was saved.

In [ ]:
response = chat("Which students are in grade 10?")
print(response)

---

## Batch INSERT

When a request contains multiple records, the Planner sets `is_batch: true`. All records are validated together before any are inserted. The Executor uses a single transaction — if one record fails, none are committed.

In [ ]:
response = chat(
    "Add these students: "
    "Jake Alvarez grade 11, Maria Chen grade 9, Devon Hill grade 12."
)
print(response)

### Query all students to confirm

A no-filter SELECT returns the full roster, showing all four students now in the system.

In [ ]:
response = chat("Show me all students.")
print(response)

---

## Different Table: Music Library

The Planner identifies the target table from context — here it routes to `music` instead of `students` and extracts the title, composer, and difficulty rating.

In [ ]:
response = chat("Add the piece Commando March by Karl King, difficulty 3.")
print(response)

In [ ]:
response = chat("Show me all pieces with difficulty 3.")
print(response)

---

## Validation

The Validator enforces business rules before executing any SQL. Only grades 9–12 are valid (high school only). The request below is rejected with a friendly explanation — no data is written to the database.

In [ ]:
response = chat("Add Alex Turner, an 8th grader, to the student list.")
print(response)

---

## Cleanup

Remove all records inserted during this demo.

In [ ]:
conn = get_connection()
try:
    with conn.cursor() as cur:
        cur.execute(
            "DELETE FROM students WHERE last_name IN (%s, %s, %s, %s)",
            ("Rodriguez", "Alvarez", "Chen", "Hill"),
        )
        cur.execute("DELETE FROM music WHERE title = %s", ("Commando March",))
    conn.commit()
    print("Demo data removed.")
except Exception as e:
    conn.rollback()
    print(f"Cleanup failed: {e}")
finally:
    conn.close()